# Task 16: Dense Retrieval Benchmark (Kaggle GPU)

**Goal**: Benchmark BAAI/bge-m3 dense retrieval against the existing BM25+source baseline.

**Environment**: Kaggle Notebook with GPU (Tesla T4 / P100).

**Output**: `task16_dense_retrieval_benchmark.json` — full metrics + per-query breakdown.

## Cell 1: Environment info

In [ ]:
import sys, platform, time, json
print(f"Python: {sys.version}")
print(f"Platform: {platform.platform()}")
try:
    import torch
    print(f"torch: {torch.__version__}")
    print(f"CUDA available: {torch.cuda.is_available()}")
    if torch.cuda.is_available():
        print(f"GPU: {torch.cuda.get_device_name(0)}")
        print(f"GPU memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
except ImportError:
    print("torch not installed")

## Cell 2: Install dependencies

In [ ]:
!pip install -q sentence-transformers faiss-gpu 2>/dev/null || pip install -q sentence-transformers faiss-cpu

import sentence_transformers, faiss, numpy
print(f"sentence-transformers: {sentence_transformers.__version__}")
print(f"faiss: {faiss.__version__}")
print(f"numpy: {numpy.__version__}")

## Cell 3: Upload JainLLM corpus files

**Before running**: Upload these files from your local `svk-corpus/` to Kaggle:

```
svk-corpus/data/release/rag_corpus.jsonl       (~418 MB)
svk-corpus/manifests/source_manifest.csv        (~15 KB)
svk-corpus/configs/retrieval_eval_queries_v1.json (~8 KB)
svk-corpus/configs/retrieval_aliases_v1.json    (~4 KB)
svk-corpus/src/svk_corpus/                      (entire src tree)
```

**Method**: Kaggle → Add Data → Upload, or drag files into the Kaggle file browser.

Set `CORPUS_DIR` below to the path where you uploaded the files.

In [ ]:
from pathlib import Path

# --- EDIT THIS PATH to match your Kaggle upload location ---
# If you upload the whole svk-corpus/ folder, this is typically:
#   /kaggle/input/jainllm-svk-corpus/svk-corpus
# or if you upload files directly:
#   /kaggle/working/svk-corpus

CORPUS_DIR = Path("/kaggle/input/jainllm-svk-corpus/svk-corpus")

# Verify files exist
required = [
    CORPUS_DIR / "data" / "release" / "rag_corpus.jsonl",
    CORPUS_DIR / "manifests" / "source_manifest.csv",
    CORPUS_DIR / "configs" / "retrieval_eval_queries_v1.json",
]
for p in required:
    status = "OK" if p.exists() else "MISSING"
    size = p.stat().st_size if p.exists() else 0
    print(f"  [{status}] {p} ({size/1e6:.1f} MB)" if size > 1e6 else f"  [{status}] {p} ({size} B)")

## Cell 4: Add svk-corpus/src to Python path

In [ ]:
import sys
src_dir = str(CORPUS_DIR / "src")
if src_dir not in sys.path:
    sys.path.insert(0, src_dir)
print(f"Added to sys.path: {src_dir}")

# Verify imports
from svk_corpus.retrieval import (
    BM25Index, SentenceTransformerDense, UnavailableDense,
    load_corpus, RetrievalResult
)
from svk_corpus.retrieval.fusion import reciprocal_rank_fusion
from svk_corpus.retrieval.rerank import Reranker
from svk_corpus.retrieval.source_channel import SourceIndex, SourceUnitExpander, build_source_index
from svk_corpus.retrieval.contract import CanonicalResult, RetrievalChannel, validate_result
print("All imports OK")

## Cell 5: Load corpus and evaluation queries

In [ ]:
import json, time

rag_path = CORPUS_DIR / "data" / "release" / "rag_corpus.jsonl"
manifest_path = CORPUS_DIR / "manifests" / "source_manifest.csv"
queries_path = CORPUS_DIR / "configs" / "retrieval_eval_queries_v1.json"

t0 = time.perf_counter()
docs = load_corpus(rag_path, manifest_path)
t_load = time.perf_counter() - t0
print(f"Corpus loaded: {len(docs)} units in {t_load:.1f}s")

queries = json.loads(queries_path.read_text(encoding="utf-8"))["queries"]
print(f"Evaluation queries: {len(queries)}")

## Cell 6: Build BM25 index

In [ ]:
t0 = time.perf_counter()
bm25 = BM25Index(docs)
t_bm25 = time.perf_counter() - t0
print(f"BM25 index built in {t_bm25:.1f}s ({len(docs)} units)")

## Cell 7: Build dense index with BAAI/bge-m3

In [ ]:
import faiss, numpy as np
from sentence_transformers import SentenceTransformer

MODEL_NAME = "BAAI/bge-m3"
BATCH_SIZE = 256  # increase if GPU memory allows
QUERY_PREFIX = "Query: "  # bge-m3 convention
DOC_PREFIX = ""

print(f"Loading model: {MODEL_NAME}")
t0 = time.perf_counter()
model = SentenceTransformer(MODEL_NAME)
t_model = time.perf_counter() - t0
print(f"Model loaded in {t_model:.1f}s")

# Encode all corpus units
texts = [DOC_PREFIX + doc.text for doc in docs]
print(f"Encoding {len(texts)} documents (batch_size={BATCH_SIZE})...")
t0 = time.perf_counter()
doc_embeddings = model.encode(
    texts,
    batch_size=BATCH_SIZE,
    show_progress_bar=True,
    normalize_embeddings=True,
    convert_to_numpy=True,
)
t_encode = time.perf_counter() - t0
print(f"Encoding complete in {t_encode:.1f}s ({t_encode/len(texts)*1000:.1f}ms/doc)")
print(f"Embedding shape: {doc_embeddings.shape}")

# Build FAISS index
dim = int(doc_embeddings.shape[1])
t0 = time.perf_counter()
index = faiss.IndexFlatIP(dim)
index.add(doc_embeddings.astype(np.float32))
t_index = time.perf_counter() - t0
print(f"FAISS index built in {t_index:.2f}s ({index.ntotal} vectors, dim={dim})")

## Cell 8: Save index artifacts (outside Git)

In [ ]:
OUT_DIR = Path("/kaggle/working/task16_artifacts")
OUT_DIR.mkdir(exist_ok=True)

faiss.write_index(index, str(OUT_DIR / "bge_m3_faiss.index"))
np.save(OUT_DIR / "doc_embeddings.npy", doc_embeddings)

# Save text_id mapping for retrieval
text_ids = [doc.text_id for doc in docs]
with open(OUT_DIR / "text_ids.json", "w") as f:
    json.dump(text_ids, f)

print(f"Artifacts saved to {OUT_DIR}")
import os
for fn in os.listdir(OUT_DIR):
    sz = os.path.getsize(OUT_DIR / fn)
    print(f"  {fn}: {sz/1e6:.1f} MB")

## Cell 9: Dense search helper

In [ ]:
def dense_search(query: str, top_k: int = 10) -> list[tuple[int, float]]:
    """Encode query, search FAISS, return (doc_index, score) pairs."""
    q_vec = model.encode(
        [QUERY_PREFIX + query],
        normalize_embeddings=True,
        convert_to_numpy=True,
    ).astype(np.float32)
    scores, indices = index.search(q_vec, min(top_k, index.ntotal))
    results = []
    for score, idx in zip(scores[0], indices[0]):
        if idx >= 0:
            results.append((int(idx), float(score)))
    return results

print("dense_search() defined")

## Cell 10: Run dense-only evaluation

In [ ]:
TOP_K = 10
RRF_K = 60

def compute_metrics(queries, hit_sources_per_q, top_k=10):
    """Compute R@5, R@10, MRR, zero-relevant for source-level queries."""
    recalls5, recalls10, rr = [], [], []
    zero = 0
    for q, hits in zip(queries, hit_sources_per_q):
        if q.get("relevance") != "source-level":
            continue
        expected = set(q.get("expected_source_ids") or [])
        if not expected:
            continue
        f5 = len(expected & set(hits[:5])) / len(expected)
        f10 = len(expected & set(hits[:top_k])) / len(expected)
        first = next((i for i, s in enumerate(hits, 1) if s in expected), None)
        recalls5.append(f5)
        recalls10.append(f10)
        rr.append(1.0 / first if first else 0.0)
        if not (expected & set(hits)):
            zero += 1
    return {
        "recall@5": round(sum(recalls5) / len(recalls5), 3) if recalls5 else None,
        "recall@10": round(sum(recalls10) / len(recalls10), 3) if recalls10 else None,
        "mrr": round(sum(rr) / len(rr), 3) if rr else None,
        "queries_with_zero_relevant": zero,
        "n_source_level": len(recalls5),
    }

print("compute_metrics() defined")

In [ ]:
# Dense-only evaluation
print("Evaluating dense-only...")
dense_hit_sources = []
dense_latencies = []
for q in queries:
    t0 = time.perf_counter()
    hits = dense_search(q["query"], top_k=TOP_K)
    dt = time.perf_counter() - t0
    dense_latencies.append(dt)
    dense_hit_sources.append([docs[idx].source_id for idx, _ in hits])

dense_metrics = compute_metrics(queries, dense_hit_sources, TOP_K)
dense_metrics["mean_latency_s"] = round(sum(dense_latencies) / len(dense_latencies), 4)
dense_metrics["max_latency_s"] = round(max(dense_latencies), 4)
print(f"  Dense-only: R@5={dense_metrics['recall@5']}  R@10={dense_metrics['recall@10']}  "
      f"MRR={dense_metrics['mrr']}  zero={dense_metrics['queries_with_zero_relevant']}  "
      f"lat={dense_metrics['mean_latency_s']}s")

## Cell 11: BM25+source baseline (regression check)

In [ ]:
print("Evaluating BM25+source baseline (C)...")
sidx = build_source_index(manifest_path)
expander = SourceUnitExpander(docs)
diversify = Reranker(strategy="diversify", max_per_source=3)

bm25_hit_sources = []
bm25_latencies = []
for q in queries:
    t0 = time.perf_counter()
    pool = bm25.search(q["query"], top_k=2000)
    unit_tier = diversify.rerank([RetrievalResult(doc=h.doc, score=h.score,
                                                   rank=h.rank, methods={"bm25": h.rank})
                                   for h in pool])[:TOP_K]
    smatches = sidx.search(q["query"], top_k=TOP_K)
    missing = [m for m in smatches
               if m.source_id not in {r.doc.source_id for r in unit_tier}
               and not m.identifier_only]
    n_fill = min(3, len(missing))
    fill = expander.expand(missing[:n_fill], top_k=n_fill)
    keep = max(TOP_K - len(fill), 0)
    combined = (list(unit_tier[:keep]) + fill)[:TOP_K]
    dt = time.perf_counter() - t0
    bm25_latencies.append(dt)
    bm25_hit_sources.append([r.doc.source_id for r in combined])

bm25_metrics = compute_metrics(queries, bm25_hit_sources, TOP_K)
bm25_metrics["mean_latency_s"] = round(sum(bm25_latencies) / len(bm25_latencies), 4)
bm25_metrics["max_latency_s"] = round(max(bm25_latencies), 4)
print(f"  BM25+src C: R@5={bm25_metrics['recall@5']}  R@10={bm25_metrics['recall@10']}  "
      f"MRR={bm25_metrics['mrr']}  zero={bm25_metrics['queries_with_zero_relevant']}  "
      f"lat={bm25_metrics['mean_latency_s']}s")

## Cell 12: Hybrid BM25+dense RRF evaluation

In [ ]:
print("Evaluating hybrid RRF (BM25+dense)...")
hybrid_hit_sources = []
hybrid_latencies = []
for q in queries:
    t0 = time.perf_counter()
    # BM25 results
    bm25_pool = bm25.search(q["query"], top_k=TOP_K * 3)
    bm25_ranks = {r.doc.text_id: i + 1 for i, r in enumerate(bm25_pool)}
    # Dense results
    dense_hits = dense_search(q["query"], top_k=TOP_K * 3)
    dense_ranks = {}
    for rank, (idx, _) in enumerate(dense_hits, 1):
        dense_ranks[docs[idx].text_id] = rank
    # RRF fusion
    all_ids = set(bm25_ranks.keys()) | set(dense_ranks.keys())
    fused = []
    for tid in all_ids:
        r1 = bm25_ranks.get(tid, TOP_K * 3 + 1)
        r2 = dense_ranks.get(tid, TOP_K * 3 + 1)
        rrf_score = 1.0 / (RRF_K + r1) + 1.0 / (RRF_K + r2)
        # Find document
        doc = None
        for r in bm25_pool:
            if r.doc.text_id == tid:
                doc = r.doc
                break
        if doc is None:
            for idx, _ in dense_hits:
                if docs[idx].text_id == tid:
                    doc = docs[idx]
                    break
        if doc is not None:
            fused.append((rrf_score, doc))
    fused.sort(key=lambda t: (-t[0], t[1].text_id))
    dt = time.perf_counter() - t0
    hybrid_latencies.append(dt)
    hybrid_hit_sources.append([doc.source_id for _, doc in fused[:TOP_K]])

hybrid_metrics = compute_metrics(queries, hybrid_hit_sources, TOP_K)
hybrid_metrics["mean_latency_s"] = round(sum(hybrid_latencies) / len(hybrid_latencies), 4)
hybrid_metrics["max_latency_s"] = round(max(hybrid_latencies), 4)
print(f"  Hybrid RRF: R@5={hybrid_metrics['recall@5']}  R@10={hybrid_metrics['recall@10']}  "
      f"MRR={hybrid_metrics['mrr']}  zero={hybrid_metrics['queries_with_zero_relevant']}  "
      f"lat={hybrid_metrics['mean_latency_s']}s")

## Cell 13: Delta analysis

In [ ]:
print("=" * 70)
print("RESULTS SUMMARY")
print("=" * 70)
for name, m in [("Dense-only", dense_metrics),
                ("BM25+src C", bm25_metrics),
                ("Hybrid RRF", hybrid_metrics)]:
    print(f"  {name:12} R@5={m['recall@5']}  R@10={m['recall@10']}  MRR={m['mrr']}  "
          f"zero={m['queries_with_zero_relevant']}  lat={m['mean_latency_s']}s")

print()
print("Delta (hybrid vs BM25 baseline):")
if bm25_metrics["recall@5"] and hybrid_metrics["recall@5"]:
    print(f"  R@5:  {hybrid_metrics['recall@5'] - bm25_metrics['recall@5']:+.3f}")
if bm25_metrics["recall@10"] and hybrid_metrics["recall@10"]:
    print(f"  R@10: {hybrid_metrics['recall@10'] - bm25_metrics['recall@10']:+.3f}")
if bm25_metrics["mrr"] and hybrid_metrics["mrr"]:
    print(f"  MRR:  {hybrid_metrics['mrr'] - bm25_metrics['mrr']:+.3f}")

## Cell 14: Per-query comparison table

In [ ]:
print(f"{'Query':<6} {'BM25 R@5':>9} {'Dense R@5':>10} {'Hybrid R@5':>11} "
      f"{'BM25 R@10':>10} {'Dense R@10':>11} {'Hybrid R@10':>12}")
print("-" * 80)
for q, b_hits, d_hits, h_hits in zip(queries, bm25_hit_sources, dense_hit_sources, hybrid_hit_sources):
    if q.get("relevance") != "source-level":
        continue
    expected = set(q.get("expected_source_ids") or [])
    if not expected:
        continue
    b5 = len(expected & set(b_hits[:5])) / len(expected)
    d5 = len(expected & set(d_hits[:5])) / len(expected)
    h5 = len(expected & set(h_hits[:5])) / len(expected)
    b10 = len(expected & set(b_hits[:10])) / len(expected)
    d10 = len(expected & set(d_hits[:10])) / len(expected)
    h10 = len(expected & set(h_hits[:10])) / len(expected)
    print(f"{q['id']:<6} {b5:>9.3f} {d5:>10.3f} {h5:>11.3f} "
          f"{b10:>10.3f} {d10:>11.3f} {h10:>12.3f}")

## Cell 15: Per-query detailed breakdown

In [ ]:
def per_query_detail(queries, hit_sources_per_q, top_k=10):
    details = []
    for q, hits in zip(queries, hit_sources_per_q):
        entry = {"id": q["id"], "category": q.get("category", ""),
                 "relevance": q.get("relevance", ""), "query": q["query"][:80]}
        if q.get("relevance") == "source-level":
            expected = set(q.get("expected_source_ids") or [])
            entry["expected"] = sorted(expected)
            entry["found_in_top5"] = sorted(expected & set(hits[:5]))
            entry["found_in_top10"] = sorted(expected & set(hits[:top_k]))
            entry["missed"] = sorted(expected - set(hits[:top_k]))
            entry["top_sources"] = hits[:top_k]
        else:
            entry["top_sources"] = hits[:5]
        details.append(entry)
    return details

dense_detail = per_query_detail(queries, dense_hit_sources)
bm25_detail = per_query_detail(queries, bm25_hit_sources)
hybrid_detail = per_query_detail(queries, hybrid_hit_sources)

# Show queries where hybrid improves over BM25
print("Queries improved by hybrid (R@5 increase):")
for q, b_hits, h_hits in zip(queries, bm25_hit_sources, hybrid_hit_sources):
    if q.get("relevance") != "source-level":
        continue
    expected = set(q.get("expected_source_ids") or [])
    if not expected:
        continue
    b5 = len(expected & set(b_hits[:5])) / len(expected)
    h5 = len(expected & set(h_hits[:5])) / len(expected)
    if h5 > b5:
        print(f"  {q['id']}: {b5:.3f} -> {h5:.3f} (+{h5-b5:.3f})  [{q['query'][:50]}]")

## Cell 16: Save full benchmark report

In [ ]:
import subprocess, os

# Collect environment info
try:
    gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none"
except:
    gpu_name = "unknown"
try:
    faiss_version = faiss.__version__
except:
    faiss_version = "unknown"

report = {
    "generated_at": time.strftime("%Y-%m-%dT%H:%M:%S"),
    "label": "task16_dense_retrieval_benchmark",
    "environment": {
        "python": sys.version,
        "torch": torch.__version__,
        "sentence_transformers": sentence_transformers.__version__,
        "faiss": faiss_version,
        "numpy": numpy.__version__,
        "gpu": gpu_name,
        "cuda_available": torch.cuda.is_available(),
    },
    "model": {
        "name": MODEL_NAME,
        "revision": "default",
        "embedding_dim": dim,
        "query_prefix": QUERY_PREFIX,
        "doc_prefix": DOC_PREFIX,
    },
    "corpus": {
        "n_units": len(docs),
        "batch_size": BATCH_SIZE,
        "encode_time_s": round(t_encode, 2),
        "index_build_time_s": round(t_index, 2),
        "model_load_time_s": round(t_model, 2),
        "ms_per_doc": round(t_encode / len(docs) * 1000, 1),
    },
    "bm25_baseline": {
        "index_build_s": round(t_bm25, 2),
    },
    "modes": {
        "dense_only": {"metrics": dense_metrics},
        "bm25_source_baseline": {"metrics": bm25_metrics},
        "hybrid_rrf": {"metrics": hybrid_metrics},
    },
    "delta_hybrid_vs_baseline": {
        "recall@5": round(hybrid_metrics["recall@5"] - bm25_metrics["recall@5"], 3)
                    if hybrid_metrics["recall@5"] and bm25_metrics["recall@5"] else None,
        "recall@10": round(hybrid_metrics["recall@10"] - bm25_metrics["recall@10"], 3)
                    if hybrid_metrics["recall@10"] and bm25_metrics["recall@10"] else None,
        "mrr": round(hybrid_metrics["mrr"] - bm25_metrics["mrr"], 3)
               if hybrid_metrics["mrr"] and bm25_metrics["mrr"] else None,
    },
    "per_query": {
        "dense_only": dense_detail,
        "bm25_source_baseline": bm25_detail,
        "hybrid_rrf": hybrid_detail,
    },
}

report_path = OUT_DIR / "task16_dense_retrieval_benchmark.json"
report_path.write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding="utf-8")
print(f"Report saved to {report_path}")
print(f"File size: {report_path.stat().st_size / 1e3:.0f} KB")

## Cell 17: Contract validation (dense results)

In [ ]:
# Validate a sample dense result against the Task 14 contract
sample_q = queries[0]
sample_hits = dense_search(sample_q["query"], top_k=3)
for idx, score in sample_hits:
    doc = docs[idx]
    cr = CanonicalResult(
        result_id=doc.text_id,
        source_id=doc.source_id,
        text=doc.text,
        title=doc.title,
        retrieval_channel=RetrievalChannel.DENSE,
        retrieval_rank=1,
        retrieval_score=score,
        dense_rank=1,
        dense_score=score,
    )
    violations = validate_result(cr)
    print(f"  {doc.text_id} (score={score:.4f}): {'VALID' if not violations else violations}")

## Cell 18: Download artifacts from Kaggle

After the notebook finishes:

1. Go to Kaggle → Output panel → download `task16_artifacts/`
2. The benchmark JSON is in `task16_artifacts/task16_dense_retrieval_benchmark.json`
3. Copy it to `svk-corpus/data/reports/` on your local machine

In [ ]:
print("Benchmark complete.")
print(f"Artifacts: {OUT_DIR}")
print("To download: Kaggle → Output → task16_artifacts/")